# Setup & Paths

In [ ]:
!pip install python-docx joblib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.6 MB/s eta 0:00:00


In [ ]:
import os
import gc
import json
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from docx import Document
from datetime import datetime

In [ ]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


In [ ]:
RAW_BASE = "/users/"
BASE_SAVE_DIR = "/users/"

DATASET_NAME = "CSE_CIC_IDS2018"
SAVE_DIR = os.path.join(BASE_SAVE_DIR, DATASET_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print("📂 RAW_BASE:", RAW_BASE)
print("💾 SAVE_DIR:", SAVE_DIR)

📂 RAW_BASE: /users/
💾 SAVE_DIR: /users/


# Load & Inspect

In [ ]:
csv_files = [
    os.path.join(RAW_BASE, f)
    for f in os.listdir(RAW_BASE)
    if f.lower().endswith(".csv")
]

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_BASE}")

print("Found CSV files:")
for f in csv_files:
    print(" -", os.path.basename(f))

dfs = []
for path in csv_files:
    print(f"📥 Loading: {os.path.basename(path)}")
    df_temp = pd.read_csv(path, low_memory=False)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

original_shape = df.shape
print("\n✅ Combined DF shape:", original_shape)
df.head()

Found CSV files:
 - DoS attacks-Slowloris.csv
 - DDOS attack-LOIC-UDP.csv
 - SQL Injection.csv
 - Brute Force -Web.csv
 - Brute Force -XSS.csv
 - DoS attacks-GoldenEye.csv
 - DoS attacks-SlowHTTPTest.csv
 - Bot.csv
 - DDOS attack-HOIC.csv
 - Infilteration.csv
 - FTP-BruteForce.csv
 - DoS attacks-Hulk.csv
 - DDoS attacks-LOIC-HTTP.csv
 - SSH-Bruteforce.csv
📥 Loading: DoS attacks-Slowloris.csv
📥 Loading: DDOS attack-LOIC-UDP.csv
📥 Loading: SQL Injection.csv
📥 Loading: Brute Force -Web.csv
📥 Loading: Brute Force -XSS.csv
📥 Loading: DoS attacks-GoldenEye.csv
📥 Loading: DoS attacks-SlowHTTPTest.csv
📥 Loading: Bot.csv
📥 Loading: DDOS attack-HOIC.csv
📥 Loading: Infilteration.csv
📥 Loading: FTP-BruteForce.csv
📥 Loading: DoS attacks-Hulk.csv
📥 Loading: DDoS attacks-LOIC-HTTP.csv
📥 Loading: SSH-Bruteforce.csv

✅ Combined DF shape: (9625148, 79)


,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,17,115999879,16,0,512,0.0,32,32,32.000000,...,8,1.100000e+07,0.000000e+00,11000000.0,11000000.0,9.182527e+06,5.757013e+06,26000000.0,5007995.0,Benign
1,80,17,119959955,14,0,448,0.0,32,32,32.000000,...,8,2.990056e+06,1.428372e+06,4000068.0,1980045.0,1.110000e+07,6.225628e+06,26000000.0,5020050.0,Benign
2,10610,6,148625,2,1,0,0.0,0,0,0.000000,...,20,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,Benign
3,22,6,1273538,2,2,0,0.0,0,0,0.000000,...,20,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,Benign
4,80,6,117252095,18,16,879,1412.0,434,0,48.833333,...,20,2.136087e+05,3.934805e+05,1399997.0,94932.0,1.000389e+07,3.400814e+04,10014176.0,9901351.0,Benign


# Label Handling

In [ ]:
print("Columns in dataset:")
print(df.columns.tolist())

label_candidates = ["Label", "label", "Attack", "attack", "Category", "class", "Class", "Label "]
label_col = None
for c in df.columns:
    if c in label_candidates:
        label_col = c
        break

if label_col is None:
    raise ValueError("❌ No label column found. Please inspect columns manually.")

print(f"\n✔ Detected label column: {label_col}")

df.rename(columns={label_col: "Label"}, inplace=True)

Columns in dataset:
['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 

In [ ]:
y_raw = df["Label"].astype(str)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

df["Label_encoded"] = y_encoded

classes = list(label_encoder.classes_)
num_classes = len(classes)
print(f"✅ Number of classes: {num_classes}")
print("Sample class mapping (index → name):")
for idx, name in enumerate(classes[:20]):
    print(f"  {idx}: {name}")
if num_classes > 20:
    print(f"... and {num_classes - 20} more classes")

# Save label encoder
label_encoder_path = os.path.join(SAVE_DIR, "label_encoder.pkl")
joblib.dump(label_encoder, label_encoder_path)
print("\n💾 Label encoder saved to:", label_encoder_path)

✅ Number of classes: 15
Sample class mapping (index → name):
  0: Benign
  1: Bot
  2: Brute Force -Web
  3: Brute Force -XSS
  4: DDOS attack-HOIC
  5: DDOS attack-LOIC-UDP
  6: DDoS attacks-LOIC-HTTP
  7: DoS attacks-GoldenEye
  8: DoS attacks-Hulk
  9: DoS attacks-SlowHTTPTest
  10: DoS attacks-Slowloris
  11: FTP-BruteForce
  12: Infilteration
  13: SQL Injection
  14: SSH-Bruteforce

💾 Label encoder saved to: /users/


# Feature Selection & Scaling

In [ ]:
X_raw = df.drop(columns=["Label", "Label_encoded"])
y = df["Label_encoded"].values

total_features_before = X_raw.shape[1]
all_feature_names_before = list(X_raw.columns)

print("🔹 TOTAL SIZE OF DATA:", df.shape[0])
print("🔹 TOTAL NUMBER OF FEATURES (before numeric filter):", total_features_before)

🔹 TOTAL SIZE OF DATA: 9625148
🔹 TOTAL NUMBER OF FEATURES (before numeric filter): 78


In [ ]:
numeric_cols = X_raw.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

X_num = X_raw[numeric_cols].copy()

print("🔹 NUMERIC FEATURE COUNT:", len(numeric_cols))
print("🔹 Example numeric features:", numeric_cols[:20])
if len(numeric_cols) > 20:
    print(f"... and {len(numeric_cols) - 20} more")

🔹 NUMERIC FEATURE COUNT: 78
🔹 Example numeric features: ['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max']
... and 58 more


In [ ]:
del X_raw
gc.collect()

0

In [ ]:
X_num = X_num.replace([np.inf, -np.inf], np.nan)
X_num = X_num.fillna(X_num.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num.values)

scaler_path = os.path.join(SAVE_DIR, "scaler.pkl")
joblib.dump(scaler, scaler_path)

print("✅ Scaling done.")
print("💾 Scaler saved to:", scaler_path)
print("🔹 SCALED SHAPE:", X_scaled.shape)

✅ Scaling done.
💾 Scaler saved to: /users/
🔹 SCALED SHAPE: (9625148, 78)


In [ ]:
selected_feature_names = numeric_cols.copy()
num_selected_features = len(selected_feature_names)
print("🔹 AFTER PREPROCESSING - NUM FEATURES:", num_selected_features)

🔹 AFTER PREPROCESSING - NUM FEATURES: 78


# Stratified 70/15/15 Split

In [ ]:
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(sss1.split(X_scaled, y))

X_train, X_temp = X_scaled[train_idx], X_scaled[temp_idx]
y_train, y_temp = y[train_idx], y[temp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(sss2.split(X_temp, y_temp))

X_val, X_test = X_temp[val_idx], X_temp[test_idx]
y_val, y_test = y_temp[val_idx], y_temp[test_idx]

print("✅ Split shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

✅ Split shapes:
Train: (6737603, 78) (6737603,)
Val  : (1443772, 78) (1443772,)
Test : (1443773, 78) (1443773,)


In [ ]:
def count_classes(y_array):
    c = Counter(y_array.tolist())
    # Convert keys to string for easier JSON/logging
    return {str(k): int(v) for k, v in c.items()}

In [ ]:
train_class_counts = count_classes(y_train)
val_class_counts   = count_classes(y_val)
test_class_counts  = count_classes(y_test)

print("\n📊 Training Class Distribution:")
print(train_class_counts)

print("\n📊 Validation Class Distribution:")
print(val_class_counts)

print("\n📊 Test Class Distribution:")
print(test_class_counts)


📊 Training Class Distribution:
{'7': 29055, '0': 4813839, '6': 403334, '1': 200334, '11': 135352, '4': 480208, '9': 97923, '14': 131312, '8': 323338, '12': 113354, '5': 1211, '10': 7693, '2': 428, '13': 61, '3': 161}

📊 Validation Class Distribution:
{'0': 1031537, '1': 42928, '6': 86428, '11': 29004, '4': 102902, '14': 28138, '8': 69287, '7': 6226, '12': 24290, '9': 20983, '10': 1649, '5': 260, '3': 35, '2': 92, '13': 13}

📊 Test Class Distribution:
{'0': 1031537, '14': 28139, '8': 69287, '9': 20984, '4': 102902, '11': 29004, '6': 86429, '1': 42929, '12': 24290, '7': 6227, '10': 1648, '2': 91, '5': 259, '3': 34, '13': 13}


# Autoencoder Definition & Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
input_dim = X_train.shape[1]
latent_dim = 64

Using device: cuda


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z

In [ ]:
ae_model = Autoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-4)
criterion = nn.SmoothL1Loss()

In [ ]:
train_tensor = torch.tensor(X_train, dtype=torch.float32)
train_dataset = TensorDataset(train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

In [ ]:
EPOCHS = 20
epoch_losses = []

for epoch in range(1, EPOCHS + 1):
    ae_model.train()
    running_loss = 0.0
    total_samples = 0

    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()

        reconstructed, _ = ae_model(batch_x)
        loss = criterion(reconstructed, batch_x)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(ae_model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * batch_x.size(0)
        total_samples += batch_x.size(0)

    avg_loss = running_loss / total_samples
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.6f}")

Epoch 1/20 - Loss: 0.009645
Epoch 2/20 - Loss: 0.003035
Epoch 3/20 - Loss: 0.002207
Epoch 4/20 - Loss: 0.001847
Epoch 5/20 - Loss: 0.001638
Epoch 6/20 - Loss: 0.001504
Epoch 7/20 - Loss: 0.001408
Epoch 8/20 - Loss: 0.001333
Epoch 9/20 - Loss: 0.001268
Epoch 10/20 - Loss: 0.001214
Epoch 11/20 - Loss: 0.001176
Epoch 12/20 - Loss: 0.001137
Epoch 13/20 - Loss: 0.001105
Epoch 14/20 - Loss: 0.001082
Epoch 15/20 - Loss: 0.001059
Epoch 16/20 - Loss: 0.001036
Epoch 17/20 - Loss: 0.001018
Epoch 18/20 - Loss: 0.001003
Epoch 19/20 - Loss: 0.000989
Epoch 20/20 - Loss: 0.000974


# Extract Latent Features

In [ ]:
def get_latent_embeddings(model, X_array, batch_size=2048):
    model.eval()
    all_z = []
    with torch.no_grad():
        for i in range(0, len(X_array), batch_size):
            batch = torch.tensor(X_array[i:i+batch_size], dtype=torch.float32).to(device)
            _, z = model(batch)
            all_z.append(z.cpu().numpy())
    return np.vstack(all_z)

print("🔁 Generating latent embeddings...")

z_train = get_latent_embeddings(ae_model, X_train)
z_val   = get_latent_embeddings(ae_model, X_val)
z_test  = get_latent_embeddings(ae_model, X_test)

print("Latent shapes:")
print("Train:", z_train.shape)
print("Val  :", z_val.shape)
print("Test :", z_test.shape)

🔁 Generating latent embeddings...
Latent shapes:
Train: (6737603, 64)
Val  : (1443772, 64)
Test : (1443773, 64)


# Save Numpy Arrays & Model

In [ ]:
np.save(os.path.join(SAVE_DIR, "train_latent.npy"), z_train)
np.save(os.path.join(SAVE_DIR, "val_latent.npy"),   z_val)
np.save(os.path.join(SAVE_DIR, "test_latent.npy"),  z_test)

np.save(os.path.join(SAVE_DIR, "y_train.npy"), y_train)
np.save(os.path.join(SAVE_DIR, "y_val.npy"),   y_val)
np.save(os.path.join(SAVE_DIR, "y_test.npy"),  y_test)

feature_list_path = os.path.join(SAVE_DIR, "feature_list.txt")
with open(feature_list_path, "w") as f:
    for col in selected_feature_names:
        f.write(col + "\n")

ae_path = os.path.join(SAVE_DIR, "autoencoder.pth")
torch.save(ae_model.state_dict(), ae_path)

print("💾 Saved:")
print(" - train_latent.npy, val_latent.npy, test_latent.npy")
print(" - y_train.npy, y_val.npy, y_test.npy")
print(" - feature_list.txt")
print(" - autoencoder.pth")

💾 Saved:
 - train_latent.npy, val_latent.npy, test_latent.npy
 - y_train.npy, y_val.npy, y_test.npy
 - feature_list.txt
 - autoencoder.pth


# Build JSON Summary

In [ ]:
label_index_to_name = {int(i): str(name) for i, name in enumerate(classes)}

summary = {
    "dataset_name": DATASET_NAME,
    "generated_at": datetime.now().isoformat(),
    "raw": {
        "num_samples": int(original_shape[0]),
        "num_features_total": int(original_shape[1]),
        "feature_names_total": all_feature_names_before,
    },
    "numeric_features": {
        "num_numeric_features": len(selected_feature_names),
        "feature_names_numeric": selected_feature_names,
    },
    "classes": {
        "num_classes": num_classes,
        "index_to_name": label_index_to_name,
    },
    "splits": {
        "train": {
            "num_samples": int(X_train.shape[0]),
            "class_counts": train_class_counts,
        },
        "val": {
            "num_samples": int(X_val.shape[0]),
            "class_counts": val_class_counts,
        },
        "test": {
            "num_samples": int(X_test.shape[0]),
            "class_counts": test_class_counts,
        },
    },
    "latent": {
        "latent_dim": int(latent_dim),
    },
}

summary_path = os.path.join(SAVE_DIR, "preprocessing_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("💾 JSON summary saved to:", summary_path)

💾 JSON summary saved to: /users/


# Create DOCX Summary

In [ ]:
doc = Document()
doc.add_heading(f"Dataset Preprocessing Summary — {DATASET_NAME}", level=1)

doc.add_paragraph(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Raw info
doc.add_heading("1. Raw Dataset Overview", level=2)
p = doc.add_paragraph()
p.add_run("Total samples: ").bold = True
p.add_run(str(original_shape[0]))
p = doc.add_paragraph()
p.add_run("Total features (before preprocessing): ").bold = True
p.add_run(str(original_shape[1]))

doc.add_paragraph("Feature names (before preprocessing):")
for name in all_feature_names_before[:50]:
    doc.add_paragraph(f"- {name}", style="List Bullet")
if len(all_feature_names_before) > 50:
    doc.add_paragraph(f"... and {len(all_feature_names_before) - 50} more features.")

# Numeric features
doc.add_heading("2. Numeric Feature Selection", level=2)
p = doc.add_paragraph()
p.add_run("Numeric features selected: ").bold = True
p.add_run(str(num_selected_features))

doc.add_paragraph("Numeric feature names:")
for name in selected_feature_names[:50]:
    doc.add_paragraph(f"- {name}", style="List Bullet")
if len(selected_feature_names) > 50:
    doc.add_paragraph(f"... and {len(selected_feature_names) - 50} more features.")

# Classes
doc.add_heading("3. Classes", level=2)
p = doc.add_paragraph()
p.add_run("Total classes: ").bold = True
p.add_run(str(num_classes))

table = doc.add_table(rows=1, cols=2)
hdr_cells = table.rows[0].cells
hdr_cells[0].text = "Class Index"
hdr_cells[1].text = "Class Name"

for idx, name in label_index_to_name.items():
    row_cells = table.add_row().cells
    row_cells[0].text = str(idx)
    row_cells[1].text = str(name)

# Splits
doc.add_heading("4. Train / Validation / Test Split", level=2)

for split_name, stats in summary["splits"].items():
    doc.add_heading(f"{split_name.upper()} Split", level=3)
    p = doc.add_paragraph()
    p.add_run("Samples: ").bold = True
    p.add_run(str(stats["num_samples"]))

    doc.add_paragraph("Class counts:")
    # Only show first N entries for readability
    items = list(stats["class_counts"].items())
    for k, v in items[:50]:
        doc.add_paragraph(f"Class {k}: {v}", style="List Bullet")
    if len(items) > 50:
        doc.add_paragraph(f"... and {len(items) - 50} more classes.", style="List Bullet")

# Latent
doc.add_heading("5. Latent Representation", level=2)
p = doc.add_paragraph()
p.add_run("Latent dimension (autoencoder): ").bold = True
p.add_run(str(latent_dim))

docx_path = os.path.join(SAVE_DIR, "preprocessing_summary.docx")
doc.save(docx_path)

print("💾 DOCX summary saved to:", docx_path)

💾 DOCX summary saved to: /users/
